# Colab Full Pipeline
Notebook ini siap untuk GitHub clone + install requirement + run pipeline.

In [ ]:
# 1) Clone repo (jika belum ada)
import os
if not os.path.isdir('mbg-sentiment'):
    !git clone https://github.com/kzquandary/mbg-sentiment.git
%cd mbg-sentiment


In [ ]:
# 2) Install dependency
!pip install -r requirements.txt

In [ ]:
# 3) Dataset check: pakai data/dataset.xlsx dari repo jika ada
from pathlib import Path
import os

Path('data').mkdir(exist_ok=True)
dataset_path = Path('data/dataset.xlsx')

if dataset_path.exists():
    print('Dataset ditemukan di repo:', dataset_path)
else:
    print('Dataset belum ada di repo. Upload dataset.xlsx sebagai fallback.')
    from google.colab import files
    up = files.upload()
    if len(up) == 0:
        raise RuntimeError('Silakan upload file dataset.xlsx')
    name = list(up.keys())[0]
    os.replace(name, str(dataset_path))
    print('Dataset tersimpan di', dataset_path)


In [ ]:
import subprocess, sys, json
import pandas as pd
from IPython.display import display, Markdown, Image

In [ ]:
def run(cmd):
    print('\n[RUN]', ' '.join(cmd))
    r = subprocess.run(cmd)
    if r.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')

steps = [
    [sys.executable, 'src/01_audit_data.py', '--input', 'data/dataset.xlsx'],
    [sys.executable, 'src/02_clean_data.py', '--input', 'data/dataset.xlsx'],
    [sys.executable, 'src/03_preprocess_text.py', '--input', 'data/cleaned_dataset.csv'],
    [sys.executable, 'src/04_eda.py', '--input', 'data/preprocessed_dataset.csv'],
    [sys.executable, 'src/05_split_data.py', '--input', 'data/preprocessed_dataset.csv'],
    [sys.executable, 'src/06_baseline_models.py'],
    [sys.executable, 'src/07_indobert_bilstm.py'],
    [sys.executable, 'src/08_tuning.py'],
    [sys.executable, 'src/09_evaluate.py'],
    [sys.executable, 'src/10_error_analysis.py'],
    [sys.executable, 'src/11_generate_report.py'],
]
for c in steps:
    run(c)
print('\nPipeline selesai.')

In [ ]:
display(Markdown('## Ringkasan Metrics'))
metrics = json.load(open('outputs/final_metrics.json', encoding='utf-8'))
display(pd.DataFrame([metrics]))

In [ ]:
display(Markdown('## Classification Report'))
display(pd.read_csv('outputs/classification_report.csv'))

In [ ]:
display(Markdown('## Confusion Matrix Figure'))
display(Image(filename='outputs/figures/confusion_matrix.png'))